<a href="https://colab.research.google.com/github/simomorphine/Humble-Systems-Theory-Reward-Free-Reinforcement-Learning/blob/main/gridworld.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import copy
import random
from collections import deque, namedtuple
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

print(f"torch {torch.__version__}, numpy {np.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


@dataclass
class Config:
    N_STATES: int = 6
    N_ACTIONS: int = 4
    HIDDEN_DIM: int = 64
    HIDDEN2_DIM: int = 64
    LR: float = 1e-3
    GAMMA: float = 0.95
    REPLAY_CAPACITY: int = 30000
    BATCH_SIZE: int = 64
    EPSILON_START: float = 1.0
    EPSILON_END: float = 0.05
    EPSILON_DECAY: int = 10000
    N_EPISODES: int = 1500
    TARGET_UPDATE: int = 200
    SEED: int = 0


Transition = namedtuple(
    "Transition", ("state", "action", "net_cost", "debt", "next_state", "done")
)

torch 2.11.0+cpu, numpy 2.1.3
CUDA available: False


In [2]:
class GridWorld:
    """
    Configurable gridworld.

    reward_mode : 'sparse' (goal only) | 'dense' (shaping)
    phi_mode    : 'distance' (to goal) | 'info' (peaks mid-grid)
    state_mode  : 'engineered' (with direction) | 'raw' (position only)
    """

    def __init__(
        self,
        size=8,
        goal=(7, 7),
        start=(0, 0),
        max_steps=200,
        reward_mode="sparse",
        phi_mode="distance",
        state_mode="engineered",
        reward_scale=10.0,
        cost_per_step=1.0,
        seed=None,
    ):
        self.size = size
        self.goal = goal
        self.start = start
        self.max_steps = max_steps
        self.reward_mode = reward_mode
        self.phi_mode = phi_mode
        self.state_mode = state_mode
        self.reward_scale = reward_scale
        self.cost_per_step = cost_per_step
        self.n_actions = 4
        self.rng = np.random.default_rng(seed)

        self.n_states = 6 if state_mode == "engineered" else 2
        self._build_phi()
        self.reset()

    def _build_phi(self):
        phi = np.zeros((self.size, self.size))
        for i in range(self.size):
            for j in range(self.size):
                if self.phi_mode == "distance":
                    phi[i, j] = (
                        abs(i - self.goal[0]) + abs(j - self.goal[1])
                    ) / (2 * self.size)
                elif self.phi_mode == "info":
                    d = (
                        abs(i - self.goal[0]) + abs(j - self.goal[1])
                    ) / (2 * self.size)
                    phi[i, j] = d * (1.0 - d)
        self.phi_grid = phi
        self.phi = phi.flatten()

    def _s2i(self, s):
        return s[0] * self.size + s[1]

    def reset(self):
        self.pos = self.start
        self.steps = 0
        self.reached_goal = False
        self.prev_phi = self.phi[self._s2i(self.pos)]
        return self._get_state()

    def _get_state(self):
        i, j = self.pos
        gi, gj = self.goal
        if self.state_mode == "engineered":
            return np.array(
                [
                    i / self.size,
                    j / self.size,
                    gi / self.size,
                    gj / self.size,
                    np.sign(gi - i),
                    np.sign(gj - j),
                ],
                dtype=np.float32,
            )
        return np.array([i / self.size, j / self.size], dtype=np.float32)

    def step(self, action):
        if self.reached_goal:
            return self._get_state(), self._terminal_info(), True

        di, dj = [(-1, 0), (1, 0), (0, -1), (0, 1)][action]
        ni = int(np.clip(self.pos[0] + di, 0, self.size - 1))
        nj = int(np.clip(self.pos[1] + dj, 0, self.size - 1))
        self.pos = (ni, nj)

        s_next = self._s2i(self.pos)
        phi_next = self.phi[s_next]

        cost = self.cost_per_step
        at_goal = self.pos == self.goal

        if at_goal:
            reward = self.reward_scale
            self.reached_goal = True
        elif self.reward_mode == "dense":
            d = (
                abs(ni - self.goal[0]) + abs(nj - self.goal[1])
            ) / (2 * self.size)
            reward = self.reward_scale * 0.1 * (1.0 - d)
        else:
            reward = 0.0

        net_cost = max(0.0, cost - reward)
        debt = phi_next - self.prev_phi
        self.prev_phi = phi_next

        self.steps += 1
        done = at_goal or (self.steps >= self.max_steps)

        return self._get_state(), {
            "cost": cost,
            "reward": reward,
            "net_cost": net_cost,
            "debt": debt,
            "reached_goal": at_goal,
        }, done

    def _terminal_info(self):
        return dict(cost=0.0, reward=0.0, net_cost=0.0, debt=0.0, reached_goal=False)

In [3]:
class StandardQNet(nn.Module):
    """Standard real-valued Q-network."""

    def __init__(self, n_states, n_actions, hidden, hidden2=None):
        super().__init__()
        hidden2 = hidden2 or hidden
        self.net = nn.Sequential(
            nn.Linear(n_states, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, n_actions),
        )
        self.n_actions = n_actions

    def forward(self, x):
        return self.net(x)


class ComplexQNet(nn.Module):
    """
    Complex-valued Q-network.

    Shared encoder, two heads: Re(Q) and Im(Q).
    Modulus |Q| = sqrt(Re^2 + Im^2).
    """

    def __init__(self, n_states, n_actions, hidden, hidden2=None):
        super().__init__()
        hidden2 = hidden2 or hidden
        self.n_actions = n_actions
        self.encoder = nn.Sequential(
            nn.Linear(n_states, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden2),
            nn.ReLU(),
        )
        self.re_head = nn.Linear(hidden2, n_actions)
        self.im_head = nn.Linear(hidden2, n_actions)

    def forward(self, x):
        f = self.encoder(x)
        return self.re_head(f), self.im_head(f)

    def modulus(self, x):
        re, im = self.forward(x)
        return torch.sqrt(re ** 2 + im ** 2 + 1e-8)

In [4]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, *args):
        self.buffer.append(Transition(*args))

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def ready(self, batch_size):
        return len(self.buffer) >= batch_size

    def __len__(self):
        return len(self.buffer)

In [5]:
class StandardDQNAgent:
    def __init__(self, cfg, device):
        self.cfg = cfg
        self.device = device
        self.name = "Standard"
        self.q_net = StandardQNet(
            cfg.N_STATES, cfg.N_ACTIONS, cfg.HIDDEN_DIM, cfg.HIDDEN2_DIM
        ).to(device)
        self.q_targ = StandardQNet(
            cfg.N_STATES, cfg.N_ACTIONS, cfg.HIDDEN_DIM, cfg.HIDDEN2_DIM
        ).to(device)
        self.q_targ.load_state_dict(self.q_net.state_dict())
        self.q_targ.eval()
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=cfg.LR)
        self.buffer = ReplayBuffer(cfg.REPLAY_CAPACITY)
        self.epsilon = cfg.EPSILON_START
        self.steps = 0

    def select_action(self, state, training=True):
        if training and random.random() < self.epsilon:
            return random.randrange(self.cfg.N_ACTIONS)
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
            q = self.q_net(s)
        return int(q.argmin(dim=-1).item())

    def update(self):
        if not self.buffer.ready(self.cfg.BATCH_SIZE):
            return None
        batch = self.buffer.sample(self.cfg.BATCH_SIZE)
        b = Transition(*zip(*batch))

        states = torch.tensor(np.array(b.state), dtype=torch.float32, device=self.device)
        actions = torch.tensor(b.action, dtype=torch.int64, device=self.device).unsqueeze(1)
        net_costs = torch.tensor(b.net_cost, dtype=torch.float32, device=self.device)
        next_states = torch.tensor(np.array(b.next_state), dtype=torch.float32, device=self.device)
        dones = torch.tensor(b.done, dtype=torch.float32, device=self.device)

        q = self.q_net(states).gather(1, actions).squeeze(1)

        with torch.no_grad():
            q_next = self.q_targ(next_states).min(dim=1)[0]
            target = net_costs + self.cfg.GAMMA * q_next * (1.0 - dones)

        loss = nn.functional.mse_loss(q, target)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_net.parameters(), 10.0)
        self.optimizer.step()
        self.steps += 1
        return loss.item()

    def update_target(self):
        self.q_targ.load_state_dict(self.q_net.state_dict())

    def decay_epsilon(self):
        self.epsilon = max(
            self.cfg.EPSILON_END,
            self.cfg.EPSILON_START
            - (self.cfg.EPSILON_START - self.cfg.EPSILON_END)
            * self.steps
            / self.cfg.EPSILON_DECAY,
        )

In [6]:
class ComplexDQNAgent:
    def __init__(self, cfg, device, zero_debt=False):
        self.cfg = cfg
        self.device = device
        self.zero_debt = zero_debt
        self.name = "Complex (zero-debt)" if zero_debt else "Complex"
        self.q_net = ComplexQNet(
            cfg.N_STATES, cfg.N_ACTIONS, cfg.HIDDEN_DIM, cfg.HIDDEN2_DIM
        ).to(device)
        self.q_targ = ComplexQNet(
            cfg.N_STATES, cfg.N_ACTIONS, cfg.HIDDEN_DIM, cfg.HIDDEN2_DIM
        ).to(device)
        self.q_targ.load_state_dict(self.q_net.state_dict())
        self.q_targ.eval()
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=cfg.LR)
        self.buffer = ReplayBuffer(cfg.REPLAY_CAPACITY)
        self.epsilon = cfg.EPSILON_START
        self.steps = 0

    def select_action(self, state, training=True):
        if training and random.random() < self.epsilon:
            return random.randrange(self.cfg.N_ACTIONS)
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
            mod = self.q_net.modulus(s)
        return int(mod.argmin(dim=-1).item())

    def update(self):
        if not self.buffer.ready(self.cfg.BATCH_SIZE):
            return None
        batch = self.buffer.sample(self.cfg.BATCH_SIZE)
        b = Transition(*zip(*batch))

        states = torch.tensor(np.array(b.state), dtype=torch.float32, device=self.device)
        actions = torch.tensor(b.action, dtype=torch.int64, device=self.device).unsqueeze(1)
        net_costs = torch.tensor(b.net_cost, dtype=torch.float32, device=self.device)
        debts = torch.tensor(b.debt, dtype=torch.float32, device=self.device)
        if self.zero_debt:
            debts = torch.zeros_like(debts)
        next_states = torch.tensor(np.array(b.next_state), dtype=torch.float32, device=self.device)
        dones = torch.tensor(b.done, dtype=torch.float32, device=self.device)

        q_re_all, q_im_all = self.q_net(states)
        q_re = q_re_all.gather(1, actions).squeeze(1)
        q_im = q_im_all.gather(1, actions).squeeze(1)

        with torch.no_grad():
            mod_next = self.q_targ.modulus(next_states)
            a_star = mod_next.argmin(dim=1, keepdim=True)
            t_re, t_im = self.q_targ(next_states)
            q_next_re = t_re.gather(1, a_star).squeeze(1)
            q_next_im = t_im.gather(1, a_star).squeeze(1)
            not_done = 1.0 - dones
            target_re = net_costs + self.cfg.GAMMA * q_next_re * not_done
            target_im = debts + self.cfg.GAMMA * q_next_im * not_done

        loss = nn.functional.mse_loss(q_re, target_re) + nn.functional.mse_loss(
            q_im, target_im
        )
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_net.parameters(), 10.0)
        self.optimizer.step()
        self.steps += 1
        return loss.item()

    def update_target(self):
        self.q_targ.load_state_dict(self.q_net.state_dict())

    def decay_epsilon(self):
        self.epsilon = max(
            self.cfg.EPSILON_END,
            self.cfg.EPSILON_START
            - (self.cfg.EPSILON_START - self.cfg.EPSILON_END)
            * self.steps
            / self.cfg.EPSILON_DECAY,
        )

    @torch.no_grad()
    def mean_abs_qim(self, state):
        s = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
        _, q_im = self.q_net(s)
        return float(q_im.abs().mean().item())

In [7]:
class TwoSelectorAgent:
    """
    Complex DQN with forward/backward selectors.

    Q^+ is updated using the backward selector on Q^-.
    Q^- is updated using the forward selector on Q^+.

    Forward selector: argmin |Q|, tie-break smallest action index.
    Backward selector: argmin |Q|, tie-break largest action index.
    """

    def __init__(self, cfg, device):
        self.cfg = cfg
        self.device = device
        self.name = "Two-Selector"
        self.q_plus = ComplexQNet(
            cfg.N_STATES, cfg.N_ACTIONS, cfg.HIDDEN_DIM, cfg.HIDDEN2_DIM
        ).to(device)
        self.q_minus = ComplexQNet(
            cfg.N_STATES, cfg.N_ACTIONS, cfg.HIDDEN_DIM, cfg.HIDDEN2_DIM
        ).to(device)
        self.q_plus_targ = copy.deepcopy(self.q_plus).to(device)
        self.q_minus_targ = copy.deepcopy(self.q_minus).to(device)
        self.q_plus_targ.eval()
        self.q_minus_targ.eval()
        self.optimizer = optim.Adam(
            list(self.q_plus.parameters()) + list(self.q_minus.parameters()),
            lr=cfg.LR,
        )
        self.buffer = ReplayBuffer(cfg.REPLAY_CAPACITY)
        self.epsilon = cfg.EPSILON_START
        self.steps = 0

    @staticmethod
    def _forward_selector(net, s):
        return net.modulus(s).argmin(dim=-1)

    @staticmethod
    def _backward_selector(net, s):
        mod = net.modulus(s)
        n_actions = mod.shape[-1]
        idx_bias = torch.arange(
            n_actions, dtype=mod.dtype, device=mod.device
        )
        # Negative bias -> largest index wins among ties
        return (mod - 1e-6 * idx_bias).argmin(dim=-1)

    def select_action(self, state, training=True):
        if training and random.random() < self.epsilon:
            return random.randrange(self.cfg.N_ACTIONS)
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
            a = self._forward_selector(self.q_plus, s)
        return int(a.item())

    def update(self):
        if not self.buffer.ready(self.cfg.BATCH_SIZE):
            return None
        batch = self.buffer.sample(self.cfg.BATCH_SIZE)
        b = Transition(*zip(*batch))

        states = torch.tensor(np.array(b.state), dtype=torch.float32, device=self.device)
        actions = torch.tensor(b.action, dtype=torch.int64, device=self.device).unsqueeze(1)
        net_costs = torch.tensor(b.net_cost, dtype=torch.float32, device=self.device)
        debts = torch.tensor(b.debt, dtype=torch.float32, device=self.device)
        next_states = torch.tensor(np.array(b.next_state), dtype=torch.float32, device=self.device)
        dones = torch.tensor(b.done, dtype=torch.float32, device=self.device)
        not_done = 1.0 - dones

        p_re_all, p_im_all = self.q_plus(states)
        m_re_all, m_im_all = self.q_minus(states)
        p_re = p_re_all.gather(1, actions).squeeze(1)
        p_im = p_im_all.gather(1, actions).squeeze(1)
        m_re = m_re_all.gather(1, actions).squeeze(1)
        m_im = m_im_all.gather(1, actions).squeeze(1)

        with torch.no_grad():
            a_minus = self._backward_selector(self.q_minus_targ, next_states).unsqueeze(1)
            tm_re, tm_im = self.q_minus_targ(next_states)
            qn_m_re = tm_re.gather(1, a_minus).squeeze(1)
            qn_m_im = tm_im.gather(1, a_minus).squeeze(1)

            a_plus = self._forward_selector(self.q_plus_targ, next_states).unsqueeze(1)
            tp_re, tp_im = self.q_plus_targ(next_states)
            qn_p_re = tp_re.gather(1, a_plus).squeeze(1)
            qn_p_im = tp_im.gather(1, a_plus).squeeze(1)

            # Q^+ uses the backward continuation, Q^- uses the forward one
            target_p_re = net_costs + self.cfg.GAMMA * qn_m_re * not_done
            target_p_im = debts + self.cfg.GAMMA * qn_m_im * not_done
            target_m_re = net_costs + self.cfg.GAMMA * qn_p_re * not_done
            target_m_im = debts + self.cfg.GAMMA * qn_p_im * not_done

        loss = (
            nn.functional.mse_loss(p_re, target_p_re)
            + nn.functional.mse_loss(p_im, target_p_im)
            + nn.functional.mse_loss(m_re, target_m_re)
            + nn.functional.mse_loss(m_im, target_m_im)
        )
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(
            list(self.q_plus.parameters()) + list(self.q_minus.parameters()), 10.0
        )
        self.optimizer.step()
        self.steps += 1
        return loss.item()

    def update_target(self):
        self.q_plus_targ.load_state_dict(self.q_plus.state_dict())
        self.q_minus_targ.load_state_dict(self.q_minus.state_dict())

    def decay_epsilon(self):
        self.epsilon = max(
            self.cfg.EPSILON_END,
            self.cfg.EPSILON_START
            - (self.cfg.EPSILON_START - self.cfg.EPSILON_END)
            * self.steps
            / self.cfg.EPSILON_DECAY,
        )

    @torch.no_grad()
    def measure_gap(self, state):
        """Gap over the actions at a single state."""
        s = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
        p_re, p_im = self.q_plus(s)
        m_re, m_im = self.q_minus(s)
        gap = (p_re - m_re) + 1j * (p_im - m_im)
        return {
            "gap_magnitude": float(gap.abs().mean().item()),
            "gap_action_std": float(gap.abs().std(dim=1).mean().item()),
            "gap_real": float((p_re - m_re).abs().mean().item()),
            "gap_imag": float((p_im - m_im).abs().mean().item()),
        }

In [8]:
def train(agent, env, cfg, log_every=100):
    n_ep = cfg.N_EPISODES
    goals = np.zeros(n_ep)
    returns = np.zeros(n_ep)
    lengths = np.zeros(n_ep)
    qim_trace = np.zeros(n_ep) if isinstance(agent, ComplexDQNAgent) else None
    gap_trace = np.zeros(n_ep) if isinstance(agent, TwoSelectorAgent) else None
    gap_real_trace = np.zeros(n_ep) if isinstance(agent, TwoSelectorAgent) else None
    gap_imag_trace = np.zeros(n_ep) if isinstance(agent, TwoSelectorAgent) else None

    for ep in range(n_ep):
        s = env.reset()
        total_net_cost = 0.0
        reached = False
        qim_sum = 0.0
        gap_samples = []
        t = 0

        for t in range(env.max_steps):
            a = agent.select_action(s, training=True)
            s_next, info, done = env.step(a)
            agent.buffer.push(s, a, info["net_cost"], info["debt"], s_next, float(done))
            agent.update()
            agent.decay_epsilon()

            total_net_cost += info["net_cost"]
            if info["reached_goal"]:
                reached = True

            if isinstance(agent, ComplexDQNAgent):
                qim_sum += agent.mean_abs_qim(s)
            if isinstance(agent, TwoSelectorAgent) and t % 10 == 0:
                gap_samples.append(agent.measure_gap(s))

            s = s_next
            if done:
                break

        if (ep + 1) % cfg.TARGET_UPDATE == 0:
            agent.update_target()

        goals[ep] = 1.0 if reached else 0.0
        returns[ep] = total_net_cost
        lengths[ep] = t + 1

        if qim_trace is not None:
            qim_trace[ep] = qim_sum / (t + 1)
        if gap_trace is not None and gap_samples:
            gap_trace[ep] = np.mean([g["gap_magnitude"] for g in gap_samples])
            gap_real_trace[ep] = np.mean([g["gap_real"] for g in gap_samples])
            gap_imag_trace[ep] = np.mean([g["gap_imag"] for g in gap_samples])

        if (ep + 1) % log_every == 0:
            msg = f"[{agent.name:18s}] ep {ep+1:4d} | goal = {goals[max(0,ep-99):ep+1].mean():.2f}"
            if gap_trace is not None:
                msg += f" | gap = {gap_trace[max(0,ep-99):ep+1].mean():.4f}"
            if qim_trace is not None:
                msg += f" | |Qim| = {qim_trace[max(0,ep-99):ep+1].mean():.4f}"
            msg += f" | eps = {agent.epsilon:.3f}"
            print(msg)

    return dict(
        goals=goals,
        returns=returns,
        lengths=lengths,
        qim=qim_trace,
        gap=gap_trace,
        gap_real=gap_real_trace,
        gap_imag=gap_imag_trace,
    )

In [9]:
def run_config(config_name, env_kwargs, n_seeds=3, n_episodes=1500, verbose=True):
    if verbose:
        print(f"\n{'='*70}\nConfig: {config_name}\n{'='*70}")

    results = {"standard": [], "complex": [], "zero_debt": [], "two_selector": []}

    for seed in range(n_seeds):
        cfg = Config()
        cfg.N_EPISODES = n_episodes
        cfg.SEED = seed

        def fresh_env():
            return GridWorld(seed=seed, **env_kwargs)

        # Standard
        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
        env = fresh_env()
        cfg_s = Config(); cfg_s.N_STATES = env.n_states; cfg_s.N_EPISODES = n_episodes
        agent = StandardDQNAgent(cfg_s, torch.device("cpu"))
        results["standard"].append(train(agent, env, cfg_s, log_every=10**9))

        # Complex
        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
        env = fresh_env()
        cfg_c = Config(); cfg_c.N_STATES = env.n_states; cfg_c.N_EPISODES = n_episodes
        agent = ComplexDQNAgent(cfg_c, torch.device("cpu"))
        results["complex"].append(train(agent, env, cfg_c, log_every=10**9))

        # Zero-debt
        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
        env = fresh_env()
        cfg_z = Config(); cfg_z.N_STATES = env.n_states; cfg_z.N_EPISODES = n_episodes
        agent = ComplexDQNAgent(cfg_z, torch.device("cpu"), zero_debt=True)
        results["zero_debt"].append(train(agent, env, cfg_z, log_every=10**9))

        # Two-selector
        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
        env = fresh_env()
        cfg_t = Config(); cfg_t.N_STATES = env.n_states; cfg_t.N_EPISODES = n_episodes
        agent = TwoSelectorAgent(cfg_t, torch.device("cpu"))
        results["two_selector"].append(train(agent, env, cfg_t, log_every=10**9))

        if verbose:
            g = {k: v[-1]["goals"][-100:].mean() for k, v in results.items()}
            print(f"  seed {seed}: " + " | ".join(f"{k}={v:.2f}" for k, v in g.items()))

    return {"config_name": config_name, "env_kwargs": env_kwargs, "results": results}

In [ ]:
configs = {
    "sparse_distance_engineered": dict(
        reward_mode="sparse", phi_mode="distance", state_mode="engineered"
    ),
    "sparse_distance_raw": dict(
        reward_mode="sparse", phi_mode="distance", state_mode="raw"
    ),
    "sparse_info_raw": dict(
        reward_mode="sparse", phi_mode="info", state_mode="raw"
    ),
    "dense_distance_engineered": dict(
        reward_mode="dense", phi_mode="distance", state_mode="engineered"
    ),
}

all_results = {}
for name, kwargs in configs.items():
    all_results[name] = run_config(name, kwargs, n_seeds=3, n_episodes=1500)


Config: sparse_distance_engineered
  seed 0: standard=1.00 | complex=1.00 | zero_debt=1.00 | two_selector=1.00
  seed 1: standard=1.00 | complex=0.97 | zero_debt=1.00 | two_selector=1.00
  seed 2: standard=1.00 | complex=1.00 | zero_debt=1.00 | two_selector=1.00

Config: sparse_distance_raw


In [ ]:
print("\n" + "=" * 95)
print("FINAL SUMMARY — goal rate, last 100 episodes (mean ± std over seeds)")
print("=" * 95)
print(f"{'Config':30s} | {'Standard':>12s} | {'Complex':>12s} | {'Zero-debt':>12s} | {'Two-Sel':>12s}")
print("-" * 95)

for name, r in all_results.items():
    res = r["results"]
    row = []
    for key in ["standard", "complex", "zero_debt", "two_selector"]:
        vals = [run["goals"][-100:].mean() for run in res[key]]
        row.append(f"{np.mean(vals):.2f}±{np.std(vals):.2f}")
    print(f"{name:30s} | {row[0]:>12s} | {row[1]:>12s} | {row[2]:>12s} | {row[3]:>12s}")

In [ ]:
def smooth(x, w):
    if len(x) < w:
        return x
    return np.convolve(x, np.ones(w) / w, mode="valid")


n_configs = len(all_results)
fig, axes = plt.subplots(1, n_configs, figsize=(6 * n_configs, 4), sharey=True)
if n_configs == 1:
    axes = [axes]

w = 50
for ax, (name, r) in zip(axes, all_results.items()):
    for key, color, label in [
        ("standard", "C0", "Standard"),
        ("complex", "C1", "Complex"),
        ("zero_debt", "C2", "Zero-debt"),
        ("two_selector", "C3", "Two-Selector"),
    ]:
        arr = np.array([run["goals"] for run in r["results"][key]])
        mean = smooth(arr.mean(axis=0), w)
        std = smooth(arr.std(axis=0), w)
        x = np.arange(len(mean))
        ax.plot(x, mean, label=label, color=color)
        ax.fill_between(x, mean - std, mean + std, alpha=0.15, color=color)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Episode")
    ax.grid(alpha=0.3)
axes[0].set_ylabel("Goal rate (smoothed)")
axes[-1].legend(loc="lower right")
plt.tight_layout()
plt.savefig("benchmark_curves.png", dpi=100)
plt.show()

In [ ]:
config_to_plot = "sparse_info_raw"
if config_to_plot in all_results:
    r = all_results[config_to_plot]["results"]["two_selector"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    w = 50

    for ax, key, title in zip(
        axes,
        ["gap", "gap_real", "gap_imag"],
        ["|ΔQ| total", "|ΔQ| real", "|ΔQ| imag"],
    ):
        arr = np.array([run[key] for run in r if run[key] is not None])
        mean = smooth(arr.mean(axis=0), w)
        std = smooth(arr.std(axis=0), w)
        x = np.arange(len(mean))
        ax.plot(x, mean, color="C3")
        ax.fill_between(x, mean - std, mean + std, alpha=0.2, color="C3")
        ax.set_title(f"{config_to_plot}\n{title}")
        ax.set_xlabel("Episode")
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("gap_analysis.png", dpi=100)
    plt.show()
else:
    print(f"Config {config_to_plot} not in results")

In [ ]:
print("\n" + "=" * 95)
print("INTERPRETATION")
print("=" * 95)

for name, r in all_results.items():
    res = r["results"]
    std = np.mean([run["goals"][-100:].mean() for run in res["standard"]])
    cpx = np.mean([run["goals"][-100:].mean() for run in res["complex"]])
    zd = np.mean([run["goals"][-100:].mean() for run in res["zero_debt"]])
    ts = np.mean([run["goals"][-100:].mean() for run in res["two_selector"]])

    print(f"\nConfig: {name}")
    print(f"  Standard  = {std:.3f}")
    print(f"  Complex   = {cpx:.3f}  (Δ vs Standard: {cpx-std:+.3f})")
    print(f"  Zero-debt = {zd:.3f}  (Δ vs Complex:  {zd-cpx:+.3f})")
    print(f"  Two-Sel   = {ts:.3f}  (Δ vs Complex:  {ts-cpx:+.3f})")

    if cpx - std > 0.05:
        print("  → Complex beats Standard")
    elif std - cpx > 0.05:
        print("  → Standard beats Complex")
    else:
        print("  → Complex ≈ Standard")

    if zd - cpx > 0.05:
        print("  → Debt channel HURTS (zero-debt better)")
    elif cpx - zd > 0.05:
        print("  → Debt channel HELPS (complex better)")
    else:
        print("  → Debt channel neutral")

    if ts - cpx > 0.05:
        print("  → Two-selector HELPS")
    elif cpx - ts > 0.05:
        print("  → Two-selector HURTS")
    else:
        print("  → Two-selector neutral")